# 05 — Treinamento da ResNet18

Este notebook **não reimplementa** o pipeline: ele chama os módulos de `src/`,
os mesmos que a linha de comando usa. A lógica mora num lugar só, então o que
você roda aqui e o que roda em `python -m src.treino` são exatamente a mesma
coisa.

O ponto metodológico central está na divisão dos dados. Cada paciente contribui
com centenas de células da mesma lâmina — mesma coloração, mesmo microscópio,
mesmo dia. Se as imagens de um paciente caírem ao mesmo tempo no treino e no
teste, o modelo pode acertar por reconhecer o paciente, não a doença. Por isso a
divisão é feita **no nível do paciente**, e a célula de dados abaixo confirma
isso explicitamente.

**Equivalente em linha de comando** (recomendado para o treino completo, porque
não depende do notebook ficar aberto):

```bash
python -m src.treino --epocas 15 --nome resnet18_paciente
```

## 1. Ambiente

In [ ]:
import sys
from pathlib import Path

# O notebook roda em notebooks/, mas os módulos são importados como `src.x`.
# Sem isso o Python não acha o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import torch

from src import config
from src.dados import calcular_pesos_classes, criar_dataloaders, resumir_divisao
from src.modelos import contar_parametros, criar_modelo
from src.treino import avaliar, obter_dispositivo, treinar
from src.utils import (
    calcular_metricas,
    definir_semente,
    formatar_metricas,
    plotar_curva_roc,
    plotar_matriz_confusao,
    plotar_curvas_treino,
)

print("Raiz do projeto:", RAIZ)
print("PyTorch:", torch.__version__)

## 2. Configuração da execução

`MODO_RAPIDO = True` roda uma amostra pequena por poucas épocas, só para
verificar que o pipeline inteiro funciona (leva ~1 min). Para o treino de
verdade, mude para `False` — são ~35 min na RTX 2070 SUPER.

In [ ]:
MODO_RAPIDO = True

if MODO_RAPIDO:
    EPOCAS, MAX_AMOSTRAS, NOME = 2, 800, "notebook_smoke_test"
else:
    EPOCAS, MAX_AMOSTRAS, NOME = config.EPOCAS, None, "resnet18_paciente_notebook"

# num_workers=0 de propósito: no Windows cada worker é um processo novo que
# reimporta o torch inteiro, e dentro do Jupyter isso costuma travar ou estourar
# o arquivo de paginação. Pela linha de comando dá para usar --workers 4.
NUM_WORKERS = 0

definir_semente(config.SEED)
dispositivo = obter_dispositivo()

print(f"\nÉpocas: {EPOCAS} | lote: {config.TAMANHO_LOTE} | semente: {config.SEED}")
if MODO_RAPIDO:
    print(f"MODO RÁPIDO: no máximo {MAX_AMOSTRAS} imagens por conjunto.")

## 3. Dados — divisão por paciente

A última linha da tabela é a verificação que sustenta toda a metodologia:
nenhum paciente pode aparecer em mais de um conjunto.

In [ ]:
loaders, divisoes = criar_dataloaders(
    tamanho_lote=config.TAMANHO_LOTE,
    semente=config.SEED,
    num_workers=NUM_WORKERS,
    estrategia="paciente",   # "aleatorio" reintroduz o vazamento — só para comparação
    max_amostras=MAX_AMOSTRAS,
)

print(resumir_divisao(divisoes))

## 4. Modelo e pesos de classe

In [ ]:
modelo = criar_modelo(arquitetura="resnet18", num_classes=len(config.CLASSES))
total, treinaveis = contar_parametros(modelo)
print(f"ResNet18 — {total:,} parâmetros ({treinaveis:,} treináveis)")

# O dataset é ~25% saudáveis / 75% leucemia. Sem correção, chutar "leucemia"
# para tudo já daria 75% de acurácia; estes pesos encarecem o erro na
# classe minoritária dentro da CrossEntropyLoss.
pesos = calcular_pesos_classes(divisoes["treino"])
print(f"Pesos: {config.CLASSES[0]}={pesos[0]:.3f}, {config.CLASSES[1]}={pesos[1]:.3f}")

## 5. Treinamento

O melhor checkpoint é escolhido pela **acurácia balanceada** de validação, não
pela acurácia simples — que num dataset 25/75 premiaria um modelo que ignora a
classe minoritária. Early stopping com paciência de 5 épocas.

In [ ]:
historico, caminho_checkpoint = treinar(
    modelo,
    loaders,
    dispositivo,
    epocas=EPOCAS,
    pesos_classes=pesos,
    paciencia=config.PACIENCIA,
    nome=NOME,
    metadados={
        "arquitetura": "resnet18",
        "estrategia_divisao": "paciente",
        "tamanho_lote": config.TAMANHO_LOTE,
        "semente": config.SEED,
        "origem": "notebook 05",
    },
)

print("Checkpoint:", caminho_checkpoint)

## 6. Curvas de treino

In [ ]:
# Distância crescente entre a curva de treino e a de validação indica overfitting.
plotar_curvas_treino(historico);

## 7. Avaliação no conjunto de teste

Recarrega os melhores pesos (e não o estado da última época) antes de medir.

In [ ]:
checkpoint = torch.load(caminho_checkpoint, map_location=dispositivo, weights_only=False)
modelo.load_state_dict(checkpoint["estado_modelo"])

_, y_verdadeiro, y_predito, y_probabilidade = avaliar(
    modelo, loaders["teste"], None, dispositivo, descricao="Teste"
)

metricas = calcular_metricas(y_verdadeiro, y_predito, y_probabilidade)
print(formatar_metricas(metricas, "Resultado no conjunto de TESTE"))

In [ ]:
plotar_matriz_confusao(y_verdadeiro, y_predito);
plotar_curva_roc(y_verdadeiro, y_probabilidade);

## 8. Próximo passo — diagnóstico por paciente

As métricas acima são por **célula**. Clinicamente o que importa é o diagnóstico
do **paciente**, que agrega as centenas de células dele numa decisão só. Isso
está em `src/avaliar.py`:

```bash
python -m src.avaliar --checkpoint outputs/resnet18_paciente_melhor.pth
```

No modelo já treinado, a agregação por paciente acerta os 30 pacientes do
conjunto de teste — desempenho bem acima do obtido célula a célula.